<a href="https://colab.research.google.com/github/ypg1um-arch/SAU_ML_TASKS/blob/main/TCGA-1_Classical_Stats_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Upload Files(.csv and R data)
from google.colab import files
uploaded = files.upload()


Saving Breast.RData to Breast.RData
Saving context1_GE.csv to context1_GE.csv
Saving context2_Meth.csv to context2_Meth.csv
Saving context3_miRNA.csv to context3_miRNA.csv
Saving context4_Protein.csv to context4_Protein.csv
Saving Table1Nature.csv to Table1Nature.csv


In [6]:
# Cell 1: Setup and Data Loading
import pandas as pd
import numpy as np

# Display settings for Pandas to show more columns during EDA
pd.set_option('display.max_columns', 50)

def load_omics_csv(filepath):
    # Read CSV and transpose so rows = samples, columns = features
    try:
        df = pd.read_csv(filepath, index_col=0)
        return df.T
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return pd.DataFrame()

# Load clinical data
clinical_df = pd.read_csv('Table1Nature.csv')
print(f"Clinical Data Shape: {clinical_df.shape}")

# Load Omics data
ge_df = load_omics_csv('context1_GE.csv')
meth_df = load_omics_csv('context2_Meth.csv')
mirna_df = load_omics_csv('context3_miRNA.csv')
prot_df = load_omics_csv('context4_Protein.csv')

print(f"Gene Expression Shape: {ge_df.shape}")
print(f"Methylation Shape: {meth_df.shape}")
print(f"miRNA Shape: {mirna_df.shape}")
print(f"Protein Shape: {prot_df.shape}")

Clinical Data Shape: (825, 30)
Gene Expression Shape: (348, 645)
Methylation Shape: (348, 574)
miRNA Shape: (348, 423)
Protein Shape: (348, 171)


In [7]:
# Cell 2: Clinical Data Summaries Using Pandas
print("--- Numeric Summaries ---")
display(clinical_df.describe().round(2))

print("\n--- Categorical Distributions ---")
# List comprehension to get value counts for categorical variables
categorical_cols = ['Gender', 'ER Status', 'PR Status', 'HER2 Final Status', 'PAM50 mRNA']

for col in categorical_cols:
    if col in clinical_df.columns:
        # Calculate raw counts and percentages purely in pandas
        counts = clinical_df[col].value_counts(dropna=False)
        pcts = clinical_df[col].value_counts(dropna=False, normalize=True) * 100
        summary_df = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.round(2)})
        print(f"\n{col}:")
        display(summary_df)

print("\n--- Missing Data Analysis ---")
missing = clinical_df.isnull().sum()
missing_pct = (missing / len(clinical_df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct (%)': missing_pct})
display(missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Pct (%)', ascending=False))

--- Numeric Summaries ---


,Age at Initial Pathologic Diagnosis,Days to date of Death,OS event,OS Time,SigClust Unsupervised mRNA,SigClust Intrinsic mRNA,miRNA Clusters,methylation Clusters,CN Clusters,Integrated Clusters (with PAM50),Integrated Clusters (no exp),Integrated Clusters (unsup exp)
count,818.00,93.00,818.00,818.00,522.00,522.00,694.00,799.00,773.00,348.00,348.00,348.00
mean,58.04,1743.75,0.11,912.26,-4.79,-6.76,4.18,3.11,2.71,2.79,2.20,2.64
std,13.21,1143.74,0.32,1072.09,3.34,4.81,1.66,1.39,1.33,0.91,1.01,1.00
min,26.00,157.00,0.00,0.00,-12.00,-13.00,1.00,1.00,1.00,1.00,1.00,1.00
25%,48.00,811.00,0.00,178.00,-7.00,-12.00,3.00,2.00,2.00,2.00,1.00,2.00
50%,58.00,1563.00,0.00,544.00,-3.00,-5.00,4.00,3.00,3.00,3.00,2.00,3.00
75%,67.00,2520.00,0.00,1293.25,-3.00,-2.00,6.00,4.00,4.00,3.00,3.00,3.00
max,90.00,4456.00,1.00,7125.00,0.00,0.00,7.00,5.00,5.00,4.00,5.00,5.00



--- Categorical Distributions ---

Gender:


,Count,Percentage (%)
Gender,,
FEMALE,810,98.18
MALE,8,0.97
NaN,7,0.85



ER Status:


...

,Count,Percentage (%)
HER2 Final Status,,
Negative,652,79.03
Positive,114,13.82
NaN,34,4.12
Not Available,15,1.82
Equivocal,10,1.21



PAM50 mRNA:


,Count,Percentage (%)
PAM50 mRNA,,
NaN,303,36.73
Luminal A,231,28.00
Luminal B,127,15.39
Basal-like,98,11.88
HER2-enriched,58,7.03
Normal-like,8,0.97



--- Missing Data Analysis ---


,Missing_Count,Missing_Pct (%)
Days to date of Death,732,88.727273
Integrated Clusters (no exp),477,57.818182
Integrated Clusters (unsup exp),477,57.818182
Integrated Clusters (with PAM50),477,57.818182
RPPA Clusters,422,51.151515
SigClust Intrinsic mRNA,303,36.727273
PAM50 mRNA,303,36.727273
SigClust Unsupervised mRNA,303,36.727273
miRNA Clusters,131,15.878788
CN Clusters,52,6.303030


In [8]:
# Cell 3: Calculating Global Statistics for Gene Expression (Pandas/Numpy)
# Instead of iterating, we use vectorized operations across the matrix

# Calculate Mean, Variance, Min, Max for every gene
omics_summary = pd.DataFrame({
    'Mean': ge_df.mean(axis=0),
    'Variance': ge_df.var(axis=0),
    'Min': ge_df.min(axis=0),
    'Max': ge_df.max(axis=0),
    'Median': ge_df.median(axis=0)
})

print("Top 10 Most Variable Genes:")
display(omics_summary.sort_values(by='Variance', ascending=False).head(10))

# Filter dataset to keep only the top 1000 most variable genes (Dimensionality Reduction via Variance)
top_1000_genes = omics_summary.nlargest(1000, 'Variance').index
ge_df_filtered = ge_df[top_1000_genes]
print(f"\nFiltered Gene Expression Shape (High Variance Only): {ge_df_filtered.shape}")

Top 10 Most Variable Genes:


,Mean,Variance,Min,Max,Median
447,-1.722206,12.251936,-10.168250,2.835250,0.0
276,0.181373,10.486483,-6.251000,8.578000,0.0
593,-0.176296,10.247764,-7.138250,5.919250,0.0
329,-0.461939,9.868949,-7.384625,5.553375,0.0
413,-0.211610,9.566383,-7.683000,6.123600,0.0
331,-0.786924,9.010733,-7.529661,4.109250,0.0
115,1.329759,7.844445,-2.951625,6.674125,0.0
365,1.262632,7.597922,-2.613000,9.817000,0.0
423,-0.917668,7.526465,-7.962958,3.008542,0.0
611,0.794482,7.523447,-2.481250,10.640625,0.0



Filtered Gene Expression Shape (High Variance Only): (348, 645)


In [10]:
# Cell 5: Differential Expression between ER Positive and ER Negative
# Here we calculate Welch's t-statistic purely using Numpy and Pandas groupby

if 'ER Status' in merged_df.columns:
    # Filter only samples with known ER status
    analysis_df = merged_df[merged_df['ER Status'].isin(['Positive', 'Negative'])]

    # Isolate just the gene columns
    gene_cols = [col for col in top_1000_genes if col in analysis_df.columns]

    # Use Pandas groupby to calculate sample sizes, means, and variances for each gene per group
    grouped = analysis_df.groupby('ER Status')[gene_cols]

    n = grouped.count().astype(float)
    means = grouped.mean()
    variances = grouped.var()

    # Extract values for ER Positive vs Negative
    mean_pos, mean_neg = means.loc['Positive'], means.loc['Negative']
    var_pos, var_neg = variances.loc['Positive'], variances.loc['Negative']
    n_pos, n_neg = n.loc['Positive'], n.loc['Negative']

    # 1. Calculate Mean Difference (Fold Change proxy if log-transformed)
    mean_diff = mean_pos - mean_neg

    # 2. Calculate Welch's T-Statistic using Numpy array operations
    # Formula: t = (M1 - M2) / sqrt((V1/n1) + (V2/n2))
    standard_error = np.sqrt((var_pos / n_pos) + (var_neg / n_neg))
    t_stat = mean_diff / standard_error

    # Compile results into a DataFrame
    stat_results = pd.DataFrame({
        'Mean_ER_Pos': mean_pos,
        'Mean_ER_Neg': mean_neg,
        'Mean_Difference': mean_diff,
        'T_Statistic': t_stat,
        'Abs_T_Statistic': np.abs(t_stat)
    })

    # Sort by the highest absolute T-Statistic (most significant difference)
    stat_results = stat_results.sort_values(by='Abs_T_Statistic', ascending=False)

    print("Top 10 Differentially Expressed Genes (ER+ vs ER-):")
    display(stat_results.head(10).round(4))

Top 10 Differentially Expressed Genes (ER+ vs ER-):


,Mean_ER_Pos,Mean_ER_Neg,Mean_Difference,T_Statistic,Abs_T_Statistic
423,0.2945,-4.7266,5.0210,21.6574,21.6574
447,-0.1879,-6.4357,6.2478,18.9888,18.9888
544,0.2825,-2.5750,2.8575,18.4472,18.4472
125,0.4428,-2.9539,3.3968,16.7187,16.7187
603,0.2099,-3.4171,3.6270,16.5391,16.5391
187,0.0413,-3.3015,3.3429,16.3540,16.3540
165,0.2906,-2.7683,3.0589,16.3431,16.3431
6,0.1763,-3.6172,3.7935,16.1760,16.1760
645,0.7603,-1.7297,2.4900,16.1059,16.1059
399,0.2133,-2.2121,2.4254,15.9910,15.9910


In [11]:
# Cell 6: Feature Correlation Analysis natively in Pandas
# Get the top 15 genes from the previous statistical test
top_15_genes = stat_results.head(15).index

# Compute Pearson Correlation Matrix using Pandas
corr_matrix = merged_df[top_15_genes].corr(method='pearson')

print("Correlation Matrix of Top 15 Differentially Expressed Genes:")
# We can use the native Pandas styler to generate a visual heatmap directly in the dataframe!
display(corr_matrix.style.background_gradient(cmap='RdBu_r', axis=None, vmin=-1, vmax=1).format("{:.2f}"))

Correlation Matrix of Top 15 Differentially Expressed Genes:


,423,447,544,125,603,187,165,6,645,399,563,381,280,185,342
423,1.00,0.85,0.86,0.73,0.68,0.81,0.81,0.73,0.72,0.83,0.75,-0.77,0.82,-0.56,0.53
447,0.85,1.00,0.77,0.76,0.69,0.80,0.77,0.75,0.66,0.80,0.77,-0.76,0.78,-0.54,0.52
544,0.86,0.77,1.00,0.70,0.62,0.75,0.69,0.70,0.66,0.75,0.72,-0.75,0.75,-0.54,0.50
125,0.73,0.76,0.70,1.00,0.62,0.69,0.67,0.66,0.62,0.63,0.71,-0.68,0.68,-0.44,0.51
603,0.68,0.69,0.62,0.62,1.00,0.64,0.60,0.60,0.56,0.64,0.63,-0.60,0.64,-0.39,0.55
187,0.81,0.80,0.75,0.69,0.64,1.00,0.75,0.69,0.64,0.80,0.75,-0.76,0.79,-0.51,0.53
165,0.81,0.77,0.69,0.67,0.60,0.75,1.00,0.75,0.74,0.78,0.67,-0.76,0.78,-0.48,0.44
6,0.73,0.75,0.70,0.66,0.60,0.69,0.75,1.00,0.66,0.72,0.67,-0.70,0.68,-0.47,0.51
645,0.72,0.66,0.66,0.62,0.56,0.64,0.74,0.66,1.00,0.70,0.65,-0.70,0.65,-0.50,0.31
399,0.83,0.80,0.75,0.63,0.64,0.80,0.78,0.72,0.70,1.00,0.74,-0.74,0.76,-0.52,0.45
